# 🗣️ Linguistic Agent — Phase 1: Whisper Transcription (Maximum Speed)

**Run on Account B — T4 GPU instance.**

Uses `datasets.Audio` + a generator to feed audio to the pipeline in large GPU batches.
The `Audio` feature loads/resamples FLAC files via soundfile (no ffmpeg crashes).

In [ ]:
!pip install -q transformers accelerate soundfile datasets pandas tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
from pathlib import Path

DRIVE_DIR       = Path('/content/drive/MyDrive/40_PER_22_Data')
TRANSCRIPTS_CSV = DRIVE_DIR / 'transcripts.csv'

# ── CONFIG ────────────────────────────────────────────────────────────────────
MAX_FILES  = 8000   # Set to None for all files
BATCH_SIZE = 32     # Reduce to 16 if you get CUDA OOM
SAVE_EVERY = 500    # Checkpoint to Drive every N files
# ──────────────────────────────────────────────────────────────────────────────

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
import pandas as pd

audio_files = []
for split in ['train', 'test']:
    for cls, label in [('bonafide', 0), ('spoof', 1)]:
        folder = DRIVE_DIR / split / cls
        if folder.exists():
            for f in sorted(folder.glob('*.flac')):
                audio_files.append({'path': str(f), 'filename': f.name,
                                    'label': label, 'split': split})

print(f'Total files found : {len(audio_files):,}')

done_files = set()
if TRANSCRIPTS_CSV.exists():
    existing   = pd.read_csv(TRANSCRIPTS_CSV)
    done_files = set(existing['filename'].tolist())
    print(f'Already done      : {len(done_files):,} — skipping')

remaining = [f for f in audio_files if f['filename'] not in done_files]
if MAX_FILES:
    remaining = remaining[:MAX_FILES]

print(f'To transcribe     : {len(remaining):,}')

In [ ]:
from datasets import Dataset, Audio as AudioFeature

# Build HF Dataset — the Audio feature uses soundfile (not ffmpeg)
# so Drive paths work fine. It also auto-resamples to 16kHz.
hf_dataset = Dataset.from_dict({
    'audio':    [f['path']     for f in remaining],
    'filename': [f['filename'] for f in remaining],
    'label':    [f['label']    for f in remaining],
    'split':    [f['split']    for f in remaining],
})
hf_dataset = hf_dataset.cast_column('audio', AudioFeature(sampling_rate=16000))
print(f'Dataset built: {len(hf_dataset):,} samples')

In [ ]:
from transformers import pipeline

print('Loading whisper-tiny...')
transcriber = pipeline(
    'automatic-speech-recognition',
    model           = 'openai/whisper-tiny',
    device          = 0 if device == 'cuda' else -1,
    generate_kwargs = {'language': 'english', 'task': 'transcribe'},
    chunk_length_s  = 30,
    torch_dtype     = torch.float16 if device == 'cuda' else torch.float32,
)
print('✅ Whisper-tiny loaded')

In [ ]:
from tqdm.auto import tqdm

# KEY FIX: The pipeline does NOT accept a Dataset directly.
# It needs a *generator* that yields individual audio dicts.
# Each sample['audio'] from a cast Audio column is already
# {'array': np.ndarray, 'sampling_rate': 16000} — exactly what
# the ASR pipeline expects. This is the correct HuggingFace pattern.
def audio_generator(dataset):
    for sample in dataset:
        yield sample['audio']

checkpoint_buf = []

for i, (sample, out) in enumerate(tqdm(
    zip(hf_dataset, transcriber(audio_generator(hf_dataset), batch_size=BATCH_SIZE)),
    total=len(hf_dataset),
    desc='Transcribing',
)):
    text = (out.get('text') or '').strip()
    checkpoint_buf.append({
        'filename': sample['filename'],
        'text':     text,
        'label':    sample['label'],
        'split':    sample['split'],
    })

    # Failsafe checkpoint
    if (i + 1) % SAVE_EVERY == 0:
        buf_df     = pd.DataFrame(checkpoint_buf)
        write_mode = 'a' if TRANSCRIPTS_CSV.exists() else 'w'
        buf_df.to_csv(TRANSCRIPTS_CSV, mode=write_mode,
                      header=(write_mode == 'w'), index=False)
        print(f'  💾 Checkpoint @ {i+1}')
        checkpoint_buf = []

# Final flush
if checkpoint_buf:
    buf_df     = pd.DataFrame(checkpoint_buf)
    write_mode = 'a' if TRANSCRIPTS_CSV.exists() else 'w'
    buf_df.to_csv(TRANSCRIPTS_CSV, mode=write_mode,
                  header=(write_mode == 'w'), index=False)

final_df = pd.read_csv(TRANSCRIPTS_CSV)
print(f'\n✅ Done!')
print(f'   Total transcripts : {len(final_df):,}')
print(f'   Non-empty text    : {(final_df["text"].str.strip() != "").sum():,}')
print(f'   Label breakdown   :\n{final_df["label"].value_counts().to_string()}')
final_df.head()